# 03 Exploratory Data Analysis

Use this notebook to explore trends, distributions, segments, anomalies, and early business signals.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
FIGURES_DIR  = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
DATA_PATH = DATA_PATH = 'https://raw.githubusercontent.com/pushkar-bit/Section-E_G-19_Healthcare-Patient-Readmission-Intelligence/main/data/processed/cleaned_hospital_readmissions.csv'
df = pd.read_csv(DATA_PATH)
df.head()

,age,time_in_hospital,n_lab_procedures,n_procedures,n_medications,n_outpatient,n_inpatient,n_emergency,medical_specialty,diag_1,...,total_visits,is_frequent_patient,stay_intensity,procedures_per_day,meds_per_visit,service_utilization_score,length_of_stay_bucket,age_numeric,age_group,is_senior
0,3,8,72,1,18,2,0,0,4,0,...,2,0,576,0,9,34.5,0,75.0,3,1
1,3,3,34,2,13,0,0,0,5,6,...,0,0,102,1,13,18.1,2,75.0,3,1
2,1,5,45,0,18,0,0,0,4,0,...,0,0,225,0,18,23.4,1,55.0,2,0
3,3,2,36,0,12,1,0,0,4,0,...,1,0,72,0,12,18.0,2,75.0,3,1
4,2,1,42,0,7,0,0,0,3,6,...,0,0,42,0,7,18.9,2,65.0,3,1


In [25]:
# Replace the example columns below with project-specific columns.
# sns.histplot(data=df, x='numeric_column')
# plt.show()
print("Shape:", df.shape)
display(df.head())

print("\nMissing values:")
display(df.isnull().sum())

Shape: (25000, 28)


,age,time_in_hospital,n_lab_procedures,n_procedures,n_medications,n_outpatient,n_inpatient,n_emergency,medical_specialty,diag_1,...,is_frequent_patient,stay_intensity,procedures_per_day,meds_per_visit,service_utilization_score,length_of_stay_bucket,age_numeric,age_group,is_senior,total_visits_bucket
0,3,8,72,1,18,2,0,0,4,0,...,0,576,0,9,34.5,0,75.0,3,1,2
1,3,3,34,2,13,0,0,0,5,6,...,0,102,1,13,18.1,2,75.0,3,1,0
2,1,5,45,0,18,0,0,0,4,0,...,0,225,0,18,23.4,1,55.0,2,0,0
3,3,2,36,0,12,1,0,0,4,0,...,0,72,0,12,18.0,2,75.0,3,1,1
4,2,1,42,0,7,0,0,0,3,6,...,0,42,0,7,18.9,2,65.0,3,1,0



Missing values:


,0
age,0
time_in_hospital,0
n_lab_procedures,0
n_procedures,0
n_medications,0
n_outpatient,0
n_inpatient,0
n_emergency,0
medical_specialty,0
diag_1,0


In [28]:
# df.describe(include='all').T

##2. Target Distribution

In [26]:
target_counts = df["readmitted_30_days"].value_counts().sort_index()
target_pct = (target_counts / len(df) * 100).round(2)

summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage (%)": target_pct
})

display(summary)

print(f"\nOverall readmission rate: {target_pct[1]}%")

,Count,Percentage (%)
readmitted_30_days,,
0,13246,52.98
1,11754,47.02



Overall readmission rate: 47.02%


## 3. Continuous Variables




In [27]:
cont_vars = [
    "time_in_hospital",
    "n_lab_procedures",
    "n_medications",
    "n_inpatient",
    "total_visits"
]

display(df[cont_vars].describe().T)

,count,mean,std,min,25%,50%,75%,max
time_in_hospital,25000.0,4.45332,3.001470,1.0,2.0,4.0,6.0,14.0
n_lab_procedures,25000.0,43.24076,19.818620,1.0,31.0,44.0,57.0,113.0
n_medications,25000.0,16.25240,8.060532,1.0,11.0,15.0,20.0,79.0
n_inpatient,25000.0,0.61596,1.177951,0.0,0.0,0.0,1.0,15.0
total_visits,25000.0,1.16896,2.150878,0.0,0.0,0.0,2.0,68.0


## 4. Outlier Analysis

In [29]:
def detect_outliers(col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df[(df[col] < lower) | (df[col] > upper)]
    return len(outliers)

for col in ["n_inpatient", "total_visits", "n_medications"]:
    print(f"{col}: {detect_outliers(col)} outliers")

n_inpatient: 1628 outliers
total_visits: 1023 outliers
n_medications: 844 outliers


##5. Trends

**📌 Trend 1: Prior Inpatient Visits**

In [38]:
trend1 = df.groupby("readmitted_30_days")["n_inpatient"].mean().round(2)
display(trend1)
print(f"Insight: Readmitted patients average {trend1[1]} visits vs {trend1[0]} for non-readmitted → clear gap.")

,n_inpatient
readmitted_30_days,
0,0.38
1,0.88


Insight: Readmitted patients average 0.88 visits vs 0.38 for non-readmitted → clear gap.


**📌 Trend 2: Length of Stay**


In [39]:
trend2 = (df.groupby("length_of_stay_bucket")["readmitted_30_days"]
          .mean() * 100).round(2)

display(trend2)

print(
    f"Insight: Readmission rises from {trend2.min()}% to {trend2.max()}% "
    f"across stay categories → longer stays indicate higher risk."
)

,readmitted_30_days
length_of_stay_bucket,
0,49.66
1,49.30
2,44.33


Insight: Readmission rises from 44.33% to 49.66% across stay categories → longer stays indicate higher risk.


**📌 Trend 3: Total Visits**

In [44]:
trend3 = (df.groupby("total_visits_bucket")["readmitted_30_days"]
          .mean() * 100).round(2)

display(trend3)

print(
    f"Insight: Readmission increases from {trend3.min()}% to {trend3.max()}% "
    f"as prior visits grow → frequent patients are at higher risk."
)

/tmp/ipykernel_32609/1581403957.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trend3 = (df.groupby("total_visits_bucket")["readmitted_30_days"]


,readmitted_30_days
total_visits_bucket,
0,37.50
1,50.93
2,58.88
3–4,62.78
5+,76.17


Insight: Readmission increases from 37.5% to 76.17% as prior visits grow → frequent patients are at higher risk.


**📌 Trend 4: Medical Specialty**

In [45]:
spec_trend = (df.groupby("medical_specialty")["readmitted_30_days"]
              .mean() * 100).round(2).sort_values(ascending=False)

display(spec_trend.head(5))

print(
    f"Insight (Department): Readmission varies across specialties, with highest at "
    f"{spec_trend.index[0]} ({spec_trend.iloc[0]}%) → operational differences across departments."
)



,readmitted_30_days
medical_specialty,
2,49.52
1,49.39
4,48.91
0,45.00
3,44.77


Insight (Department): Readmission varies across specialties, with highest at 2 (49.52%) → operational differences across departments.


##6. Categorical Relationships

In [41]:
df.groupby("medical_specialty")["readmitted_30_days"].mean().sort_values(ascending=False)

,readmitted_30_days
medical_specialty,
2,0.495218
1,0.493899
4,0.489097
0,0.449965
3,0.447686
5,0.414790
6,0.412201


##**Summary - Table**

In [ ]:
summary_data = [
    {
        "Area": "Target",
        "Finding": f"{target_pct[1]}% patients are readmitted within 30 days",
        "Insight": "Mild class imbalance present"
    },
    {
        "Area": "Prior Inpatient Visits",
        "Finding": f"{trend1[1]} vs {trend1[0]} avg visits (Readmitted vs Not)",
        "Insight": "Strongest predictor of readmission"
    },
    {
        "Area": "Length of Stay",
        "Finding": f"{trend2.min()}% → {trend2.max()}% readmission range",
        "Insight": "Longer stays indicate higher risk"
    },
    {
        "Area": "Total Visits",
        "Finding": f"{trend3.min()}% → {trend3.max()}% readmission range",
        "Insight": "Frequent patients are high-risk"
    },
    {
        "Area": "Correlation",
        "Finding": top_corr.index[0],
        "Insight": "Most correlated feature with readmission"
    },
    {
        "Area": "Outliers",
        "Finding": "High-utilisation patients detected",
        "Insight": "Represents critical intervention group"
    }
]

summary_df = pd.DataFrame(summary_data)
display(summary_df)